# Assignment 24: Titanic Classification & Clustering Analysis

**Objective**: Apply multiple Machine Learning algorithms to the Titanic dataset.
- **Supervised**: Naive Bayes, Decision Tree, ANN, Linear Regression
- **Unsupervised**: K-Means, DBSCAN, Self-Organizing Map (SOM)

This notebook demonstrates data preprocessing, model training, and visualization of results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, r2_score

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LinearRegression

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

In [ ]:
# --- 1. Data Loading & Preprocessing ---

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
try:
    df = pd.read_csv(url)
    print("Titanic dataset loaded from URL.")
except:
    print("Network error loading Titanic. Please check internet connection.")
    # Fallback simulation for verification if offline (though usually notebook is run online)
    df = pd.DataFrame({
        'Survived': np.random.randint(0, 2, 100),
        'Pclass': np.random.randint(1, 4, 100),
        'Sex': np.random.choice(['male', 'female'], 100),
        'Age': np.random.rand(100) * 80,
        'SibSp': np.random.randint(0, 5, 100),
        'Parch': np.random.randint(0, 5, 100),
        'Fare': np.random.rand(100) * 100,
        'Embarked': np.random.choice(['S', 'C', 'Q'], 100)
    })

# Target & Features
y = df["Survived"].astype(int)
feature_cols = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[feature_cols].copy()

# Identify columns
numeric_cols = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_cols = ["Sex", "Embarked"]

# Pipeline: Impute + Scale/Encode
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols)
    ],
    remainder="drop"
)

# Train-test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Data Shape: {df.shape}")
print(f"Train: {X_tr.shape}, Test: {X_te.shape}")

In [ ]:
# --- 2. Classification Models ---

models = {
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(criterion="entropy", max_depth=6, min_samples_leaf=8, random_state=42),
    "ANN (MLP)": MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", max_iter=500, random_state=42)
}

results = {}

for name, clf in models.items():
    # Create full pipeline for each model
    pipeline = Pipeline(steps=[("prep", preprocess), ("clf", clf)])
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    results[name] = acc
    
    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_te, y_pred, target_names=["Died", "Survived"]))
    
    # Confusion Matrix
    cm = confusion_matrix(y_te, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Died", "Survived"])
    disp.plot(cmap='Blues')
    plt.title(f"{name} Confusion Matrix")
    plt.show()

# Linear Regression as Classifier
print("=== Linear Regression (Threshold 0.5) ===")
lr_pipe = Pipeline(steps=[("prep", preprocess), ("reg", LinearRegression())])
lr_pipe.fit(X_tr, y_tr)
y_cont = lr_pipe.predict(X_te)
y_lr_pred = (y_cont >= 0.5).astype(int)

acc_lr = accuracy_score(y_te, y_lr_pred)
mse_lr = mean_squared_error(y_te, y_cont)
results["Linear Regression"] = acc_lr

print(f"Accuracy: {acc_lr:.4f}")
print(f"Mean Squared Error: {mse_lr:.4f}")

plt.figure(figsize=(8, 4))
plt.scatter(y_te, y_cont, alpha=0.6)
plt.plot([0, 1], [0, 1], color="red", linestyle="--")
plt.xlabel("Actual")
plt.ylabel("Predicted Probability")
plt.title("Linear Regression Predictions")
plt.show()

In [ ]:
# --- 3. Clustering Models (Unsupervised) ---

# Prepare Data for Clustering (Use Scale/Encoded Data)
X_all_proc = preprocess.fit_transform(X)
# Dense conversion if sparse
if hasattr(X_all_proc, "toarray"):
    X_all_proc = X_all_proc.toarray()

# PCA for Visualization (2D)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_all_proc)

# K-Means
print("=== K-Means Clustering ===")
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(X_all_proc)
print(f"Silhouette Score: {silhouette_score(X_all_proc, km_labels):.4f}")

plt.figure(figsize=(8, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap='viridis', alpha=0.6)
plt.title("K-Means Clusters (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label='Cluster')
plt.show()

# DBSCAN
print("=== DBSCAN Clustering ===")
dbscan = DBSCAN(eps=1.5, min_samples=5)
db_labels = dbscan.fit_predict(X_all_proc)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)
print(f"Clusters: {n_clusters}, Noise points: {n_noise}")

plt.figure(figsize=(8, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=db_labels, cmap='plasma', alpha=0.6)
plt.title("DBSCAN Clusters (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label='Cluster')
plt.show()

In [ ]:
# --- 4. Self-Organizing Map (SOM) ---

print("=== Self-Organizing Map (SOM) ===")

def train_som(X, grid=(10, 10), lr=0.5, sigma=3.0, epochs=10, random_state=42):
    rng = np.random.default_rng(random_state)
    n_samples, n_features = X.shape
    gx, gy = grid
    
    # Initialize weights randomly
    W = rng.normal(0, 1, size=(gx, gy, n_features))
    
    # Coordinate grid for neighborhood calc
    coords = np.array([(i, j) for i in range(gx) for j in range(gy)]).reshape(gx, gy, 2)
    
    for epoch in range(epochs):
        # Decay rates
        lr_t = lr * (1 - epoch / epochs)
        sigma_t = sigma * (1 - epoch / epochs)
        if sigma_t < 0.1: sigma_t = 0.1
        
        # Shuffle data
        indices = rng.permutation(n_samples)
        for idx in indices:
            x = X[idx]
            
            # Best Matching Unit (BMU)
            dists = np.linalg.norm(W - x, axis=2)
            bmu = np.unravel_index(np.argmin(dists), (gx, gy))
            
            # Neighborhood function (Gaussian)
            # Only update neighbors within sigma radius efficiency hack can be applied here,
            # but full grid update is cleaner for small grids
            dist_sq = np.sum((coords - np.array(bmu))**2, axis=2)
            h = np.exp(-dist_sq / (2 * sigma_t**2))
            
            # Update weights
            # W += lr * h * (x - W) -> broadcast x-W
            W += lr_t * h[..., None] * (x - W)
            
    return W

def get_som_hits(X, W):
    gx, gy, _ = W.shape
    hits = np.zeros((gx, gy))
    for x in X:
        dists = np.linalg.norm(W - x, axis=2)
        bmu = np.unravel_index(np.argmin(dists), (gx, gy))
        hits[bmu] += 1
    return hits

# Train SOM
som_grid = (10, 10)
weights = train_som(X_all_proc, grid=som_grid, epochs=20)
hit_map = get_som_hits(X_all_proc, weights)

# Visualization
plt.figure(figsize=(8, 6))
plt.imshow(hit_map, cmap='hot', interpolation='nearest')
plt.title("SOM Hit Map (Passenger Density)")
plt.colorbar(label="Count")
plt.show()